# SimCLR: Contrastive Learning from Scratch

## An Interactive Tutorial

This notebook demonstrates **SimCLR** (Simple Framework for Contrastive Learning of Visual Representations), a groundbreaking self-supervised learning method.

---

## What is Contrastive Learning?

**Traditional Supervised Learning:**
```
Image → [Model] → Prediction → Compare to Label → Update
```
Problem: Requires expensive labeled data!

**Contrastive Learning:**
```
Image → [Augment] → View1, View2 → [Model] → Embeddings → Compare Embeddings → Update
```
Key insight: No labels needed! Learn by **comparing examples**.

---

## The SimCLR Approach

### 1. Create Positive Pairs via Augmentation

For each image, create two augmented views:
- Random crop + resize
- Color jitter
- Random grayscale
- Random horizontal flip
- Gaussian blur

These two views are a **positive pair** - they look different but represent the same semantic content.

### 2. Contrastive Loss (NT-Xent)

For a batch of N images with 2N augmented views:

$$\mathcal{L}_{i,j} = -\log\frac{\exp(\text{sim}(z_i, z_j)/\tau)}{\sum_{k \neq i} \exp(\text{sim}(z_i, z_k)/\tau)}$$

Where:
- $z_i, z_j$ = embeddings of a positive pair
- $\tau$ = temperature parameter
- $\text{sim}$ = cosine similarity

**Goal:** Pull positive pairs together, push all negatives apart.

### 3. Architecture

```
Image → Encoder (ResNet) → Representation → Projection Head → Embedding → NT-Xent Loss
                              ↑                                        
                         (keep this)                              (discard after training)
```

The projection head is **removed** after pretraining - we only keep the encoder!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from sklearn.decomposition import PCA

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

---

## Part 1: Understanding Data Augmentation

Let's see how SimCLR creates positive pairs through augmentation.

In [ ]:
# Define SimCLR-style augmentation
def get_simclr_transform(input_size=32):
    return transforms.Compose([
        transforms.RandomResizedCrop(input_size, scale=(0.2, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
        transforms.RandomGrayscale(p=0.2),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

# Load CIFAR-10
base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=base_transform
)

print(f"Dataset size: {len(train_dataset)} images")

In [ ]:
# Visualize augmented pairs
def visualize_augmentation_pairs(dataset, n_pairs=5):
    """Show how augmentations create positive pairs"""
    simclr_transform = get_simclr_transform()
    
    fig, axes = plt.subplots(n_pairs, 3, figsize=(9, 2.5*n_pairs))
    
    class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']
    
    for i in range(n_pairs):
        # Get original image
        img, label = dataset[i + 10]
        
        # Create two augmented views
        view1 = simclr_transform(img)
        view2 = simclr_transform(img)
        
        # Convert for display
        orig_np = img.permute(1, 2, 0).numpy() * 0.5 + 0.5
        v1_np = view1.permute(1, 2, 0).numpy() * 0.5 + 0.5
        v2_np = view2.permute(1, 2, 0).numpy() * 0.5 + 0.5
        
        orig_np = np.clip(orig_np, 0, 1)
        v1_np = np.clip(v1_np, 0, 1)
        v2_np = np.clip(v2_np, 0, 1)
        
        axes[i, 0].imshow(orig_np)
        axes[i, 0].set_title(f'Original: {class_names[label]}')
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(v1_np)
        axes[i, 1].set_title('Augmented View 1')
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(v2_np)
        axes[i, 2].set_title('Augmented View 2')
        axes[i, 2].axis('off')
    
    plt.suptitle('SimCLR Positive Pairs: Same Image, Different Augmentations', fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_augmentation_pairs(train_dataset)

**Key Observation:** Views 1 and 2 look quite different (different crops, colors, flips) but represent the **same underlying object**. The model must learn to recognize them as semantically identical!

---

## Part 2: The NT-Xent Contrastive Loss

This is the core of SimCLR. Let's implement it step by step.

In [ ]:
class NTXentLoss(nn.Module):
    """
    NT-Xent: Normalized Temperature-scaled Cross Entropy Loss
    
    Visual explanation:
    
    Batch of 4 images → 8 augmented views (2 per image)
    
    Views:     [v1_1, v1_2, v2_1, v2_2, v3_1, v3_2, v4_1, v4_2]
    Positive:   (v1_1↔v1_2) (v2_1↔v2_2) (v3_1↔v3_2) (v4_1↔v4_2)
    
    For view v1_1:
    - Positive: v1_2 (same original image)
    - Negatives: v2_1, v2_2, v3_1, v3_2, v4_1, v4_2 (different images)
    
    Loss encourages:
    - High similarity between positives
    - Low similarity between negatives
    """
    
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, z_i, z_j):
        """
        Args:
            z_i: Embeddings of view 1 [N, D]
            z_j: Embeddings of view 2 [N, D]
        
        Returns:
            loss: NT-Xent loss (scalar)
        """
        N = z_i.shape[0]
        
        # Step 1: L2 normalize (converts dot product to cosine similarity)
        z_i = F.normalize(z_i, p=2, dim=1)
        z_j = F.normalize(z_j, p=2, dim=1)
        
        # Step 2: Concatenate all embeddings → [2N, D]
        all_z = torch.cat([z_i, z_j], dim=0)
        
        # Step 3: Compute similarity matrix [2N, 2N]
        # sim[a, b] = cosine similarity between embedding a and b
        sim_matrix = torch.matmul(all_z, all_z.T)
        
        # Step 4: Scale by temperature
        # Lower temp = more confident/p peaked distribution
        sim_matrix = sim_matrix / self.temperature
        
        # Step 5: Create labels for positive pairs
        # For view i (0..N-1), positive is at (i + N)
        # For view i (N..2N-1), positive is at (i - N)
        labels = torch.cat([torch.arange(N, device=z_i.device) + N, 
                            torch.arange(N, device=z_i.device)])
        
        # Step 6: Cross entropy loss
        # Maximizes similarity of positive pairs relative to negatives
        loss = F.cross_entropy(sim_matrix, labels)
        
        return loss

# Test the loss function
criterion = NTXentLoss(temperature=0.5)

# Random embeddings
z_i = torch.randn(32, 64)  # 32 images, 64-D embeddings
z_j = torch.randn(32, 64)

loss = criterion(z_i, z_j)
print(f"NT-Xent Loss (random embeddings): {loss.item():.4f}")

# Perfect positive pairs (should give very low loss)
z_perfect = torch.randn(32, 64)
loss_perfect = criterion(z_perfect, z_perfect)  # Identical = perfect positive
print(f"NT-Xent Loss (identical pairs): {loss_perfect.item():.4f}")

---

## Part 3: Building the SimCLR Model

SimCLR uses:
1. **Encoder**: Extracts features (ResNet backbone)
2. **Projection Head**: MLP that transforms features for the contrastive loss

In [ ]:
class SimCLR(nn.Module):
    """
    SimCLR Model Architecture
    
    Image → Encoder → Representation → Projection Head → Embedding
                                         ↑                    ↑
                                    (remove after)      (NT-Xent loss here)
                                     training
    """
    
    def __init__(self, input_dim=512, hidden_dim=512, output_dim=128):
        super().__init__()
        
        # Encoder: Pretrained ResNet-18 (without final FC layer)
        resnet = torchvision.models.resnet18(weights=None)
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])
        
        # Projection Head: 2-layer MLP
        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        """
        Returns:
            representation: Features from encoder [B, 512]
            embedding: Output of projection head [B, 128]
        """
        representation = self.encoder(x).squeeze(-1).squeeze(-1)
        embedding = self.projection(representation)
        return representation, embedding

# Count parameters
model = SimCLR()
total_params = sum(p.numel() for p in model.parameters())
print(f"\nSimCLR Model Parameters: {total_params:,}")

# Show architecture
print("\nModel Architecture:")
print(model)

---

## Part 4: Training Loop

Key aspects of SimCLR training:
1. **No labels used** - purely self-supervised
2. **Large batch sizes** - more negatives = better learning
3. **Two forward passes** - one for each augmented view

In [ ]:
# Prepare dataset with SimCLR augmentations
class SimCLRDataset(torch.utils.data.Dataset):
    """Wrapper that applies two random augmentations per image"""
    
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset
        self.transform = get_simclr_transform()
    
    def __getitem__(self, idx):
        img, label = self.base_dataset[idx]
        # Create two augmented views (positive pair)
        view1 = self.transform(img)
        view2 = self.transform(img)
        return (view1, view2), label  # label not used in training!
    
    def __len__(self):
        return len(self.base_dataset)

simclr_dataset = SimCLRDataset(train_dataset)

# IMPORTANT: Large batch size for many negative samples
batch_size = 256
train_loader = DataLoader(simclr_dataset, batch_size=batch_size, 
                          shuffle=True, num_workers=0, drop_last=True)

print(f"Training samples: {len(simclr_dataset)}")
print(f"Batch size: {batch_size}")
print(f"Negative samples per batch: {2 * batch_size - 2}")

In [ ]:
def train_simclr(model, dataloader, criterion, optimizer, device, n_epochs):
    """
    Train SimCLR model using contrastive learning
    
    For each batch:
    1. Get two augmented views for each image
    2. Pass both through encoder + projection head
    3. Compute NT-Xent loss
    4. Backpropagate
    """
    
    history = {'loss': [], 'pos_sim': [], 'neg_sim': []}
    
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        epoch_pos_sim = 0
        epoch_neg_sim = 0
        n_batches = 0
        
        pbar = tqdm(dataloader, desc=f'Epoch {epoch+1}/{n_epochs}')
        
        for (view1, view2), _ in pbar:
            view1, view2 = view1.to(device), view2.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass through both views
            _, embed1 = model(view1)
            _, embed2 = model(view2)
            
            # Compute contrastive loss
            loss = criterion(embed1, embed2)
            
            loss.backward()
            optimizer.step()
            
            # Track metrics
            with torch.no_grad():
                all_embeds = torch.cat([embed1, embed2], dim=0)
                all_embeds = F.normalize(all_embeds, p=2, dim=1)
                sim_matrix = torch.matmul(all_embeds, all_embeds.T)
                
                N = embed1.shape[0]
                pos_sim = torch.diag(sim_matrix[:N, N:]).mean().item()
                
                # Sample some negatives
                neg_samples = []
                for i in range(min(N, 10)):
                    for j in range(N):
                        if i != j:
                            neg_samples.append(sim_matrix[i, j+N].item())
                neg_sim = np.mean(neg_samples) if neg_samples else 0
                
                epoch_pos_sim += pos_sim
                epoch_neg_sim += neg_sim
            
            epoch_loss += loss.item()
            n_batches += 1
            
            pbar.set_postfix({'loss': f'{loss.item():.3f}'})
        
        # Average metrics
        avg_loss = epoch_loss / n_batches
        avg_pos_sim = epoch_pos_sim / n_batches
        avg_neg_sim = epoch_neg_sim / n_batches
        
        history['loss'].append(avg_loss)
        history['pos_sim'].append(avg_pos_sim)
        history['neg_sim'].append(avg_neg_sim)
        
        print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, "
              f"PosSim={avg_pos_sim:.3f}, NegSim={avg_neg_sim:.3f}")
    
    return history

# Training setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")

model = SimCLR().to(device)
criterion = NTXentLoss(temperature=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-6)

# For demo, train for fewer epochs (increase to 50-100 for real use)
n_epochs = 10

print(f"\nStarting SimCLR training for {n_epochs} epochs...")
print("This is self-supervised - NO LABELS ARE USED!")
print("-" * 50)

history = train_simclr(model, train_loader, criterion, optimizer, device, n_epochs)

---

## Part 5: Visualizing Results

Let's see what the model learned!

In [ ]:
def plot_training_history(history):
    """Plot training curves"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Loss
    axes[0].plot(history['loss'], 'b-', linewidth=2)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('NT-Xent Loss')
    axes[0].set_title('Training Loss')
    axes[0].grid(True, alpha=0.3)
    
    # Similarity
    axes[1].plot(history['pos_sim'], 'g-', linewidth=2, label='Positive Pairs')
    axes[1].plot(history['neg_sim'], 'r-', linewidth=2, label='Negative Pairs')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Cosine Similarity')
    axes[1].set_title('Pairwise Similarity')
    axes[1].set_ylim(-0.2, 1.0)
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    # Gap
    gap = [p - n for p, n in zip(history['pos_sim'], history['neg_sim'])]
    axes[2].plot(gap, 'purple', linewidth=2)
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Similarity Gap')
    axes[2].set_title('Positive - Negative Similarity\n(Higher = Better Separation)')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('simclr_history.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_training_history(history)

In [ ]:
def visualize_embeddings(model, dataloader, device, n_samples=500):
    """
    Visualize learned embeddings using PCA
    
    Even though we didn't use labels, do the embeddings cluster by class?
    """
    model.eval()
    
    all_reps = []
    all_labels = []
    
    with torch.no_grad():
        for (view1, view2), labels in dataloader:
            view1 = view1.to(device)
            reps, _ = model(view1)
            all_reps.append(reps.cpu())
            all_labels.append(labels)
            
            if len(all_reps) * reps.shape[0] >= n_samples:
                break
    
    reps_np = torch.cat(all_reps, dim=0)[:n_samples].numpy()
    labels_np = torch.cat(all_labels, dim=0)[:n_samples].numpy()
    
    # PCA to 2D
    pca = PCA(n_components=2)
    reps_2d = pca.fit_transform(reps_np)
    
    # Plot
    plt.figure(figsize=(10, 8))
    
    class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']
    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    
    for class_idx in range(10):
        mask = labels_np == class_idx
        plt.scatter(reps_2d[mask, 0], reps_2d[mask, 1],
                   c=[colors[class_idx]], alpha=0.5, s=30,
                   label=class_names[class_idx])
    
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
    plt.title('PCA of Learned Representations\n'
              'Colors = true labels (NOT used during training!)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.savefig('simclr_pca.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_embeddings(model, train_loader, device)

---

## Part 6: Evaluating with Linear Probing

To measure representation quality:
1. **Freeze** the encoder
2. Train a **linear classifier** on top
3. Evaluate on test set

This is called **linear probing** and measures how good the learned features are.

In [ ]:
def linear_probe_eval(model, train_loader, test_loader, device, n_epochs=10):
    """
    Evaluate representations by training a linear classifier
    """
    print("\n" + "="*50)
    print("Linear Probing Evaluation")
    print("="*50)
    
    # Freeze encoder
    model.encoder.eval()
    for param in model.encoder.parameters():
        param.requires_grad = False
    
    # Linear classifier
    classifier = nn.Linear(512, 10).to(device)
    optimizer = torch.optim.Adam(classifier.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    # Extract features once (since encoder is frozen)
    print("Extracting features...")
    
    train_features = []
    train_labels = []
    
    model.eval()
    with torch.no_grad():
        for (view1, view2), labels in train_loader:
            view1 = view1.to(device)
            reps, _ = model(view1)
            train_features.append(reps.cpu())
            train_labels.append(labels)
    
    X_train = torch.cat(train_features, dim=0)
    y_train = torch.cat(train_labels, dim=0)
    
    # Training loop
    train_dataset = TensorDataset(X_train, y_train)
    feature_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
    
    for epoch in range(n_epochs):
        classifier.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for features, labels in feature_loader:
            features, labels = features.to(device), labels.to(device)
            
            optimizer.zero_grad()
            logits = classifier(features)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = logits.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_acc = 100. * correct / total
        print(f"Epoch {epoch+1}/{n_epochs}: Loss={total_loss/len(feature_loader):.4f}, "
              f"Train Acc={train_acc:.1f}%")
    
    # Test evaluation
    print("\nEvaluating on test set...")
    
    test_features = []
    test_labels = []
    
    with torch.no_grad():
        for (view1, view2), labels in test_loader:
            view1 = view1.to(device)
            reps, _ = model(view1)
            test_features.append(reps.cpu())
            test_labels.append(labels)
    
    X_test = torch.cat(test_features, dim=0)
    y_test = torch.cat(test_labels, dim=0)
    
    classifier.eval()
    with torch.no_grad():
        logits = classifier(X_test.to(device))
        _, predicted = logits.max(1)
        test_acc = 100. * predicted.eq(y_test.to(device)).sum().item() / len(y_test)
    
    print(f"\n{'='*50}")
    print(f"TEST ACCURACY: {test_acc:.2f}%")
    print(f"{'='*50}")
    print(f"""
Note: This accuracy is after limited training (10 epochs).
In the SimCLR paper, they achieve ~93% on CIFAR-10 with:
- 100+ epochs of contrastive pretraining
- Larger batch sizes (512+)
- Stronger augmentations
""")
    
    # Unfreeze encoder for cleanup
    for param in model.encoder.parameters():
        param.requires_grad = True
    
    return test_acc

# Create test loader
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=base_transform
)
test_dataset_simclr = SimCLRDataset(test_dataset)
test_loader = DataLoader(test_dataset_simclr, batch_size=batch_size, shuffle=False, num_workers=0)

# Run linear probe
test_acc = linear_probe_eval(model, train_loader, test_loader, device)

---

## Summary: Why SimCLR Works

### Key Insights

1. **Self-Supervised Learning**
   - No human labels needed
   - The data itself provides the supervision signal

2. **Contrastive Objective**
   - Learn by comparing examples
   - Similar things → similar embeddings
   - Different things → different embeddings

3. **Data Augmentation is Crucial**
   - Creates challenging positive pairs
   - Forces model to learn invariant features
   - Model can't cheat using superficial cues

4. **Large Batch Sizes**
   - More negative samples per batch
   - Better contrastive signal

5. **Projection Head**
   - Transforms features for contrastive loss
   - Removed after training
   - Helps backbone learn better representations

### What Does the Model Learn?

The model learns **invariant features**:
- Shape and structure (preserved across augmentations)
- Object parts and textures
- Semantic similarity

The model ignores **superficial features**:
- Exact color (changed by color jitter)
- Exact position (changed by random crop)
- Orientation (changed by flips)

These invariant features are exactly what we want for downstream tasks!

### Comparison to Supervised Learning

| Aspect | Supervised | SimCLR |
|--------|------------|--------|
| Labels | Required | Not used |
| Signal | Human annotations | Data augmentations |
| Cost | Expensive | Free |
| Features | Task-specific | General purpose |

---

## Exercises

Try modifying these parameters to see their effect:

1. **Temperature**: Change from 0.5 to 0.1 or 1.0
2. **Batch size**: Try 64, 128, 512
3. **Augmentations**: Remove color jitter or blur
4. **Embedding size**: Try 32, 64, 256
5. **Training epochs**: Train longer for better results